# 09a - Do the multimodal features make PLOS predictable on Synthea? (run_02)

Real data only, no synthetic rows. Same classifier, same patient split (train / val / test from notebook 01).
Compares average precision (AP) on test for:
- the tabular features used so far (notebook 01),
- the 524-number vector from notebook 07, block by block (profile, conditions, procedures, notes, medications, flags),
- everything combined.

If AP rises clearly above the tabular baseline, Synthea becomes usable for measuring augmentation gain on these features.
**Caution:** notes, medications, procedures, vitals and the activity flags come from DURING the stay. A large jump may
partly be information that is not available at admission. Cell 4 reports an admission-safe subset separately.

Runtime: CPU, about 5-10 minutes.

In [ ]:
# Cell 1 - load
import os, sys, json, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
if "google.colab" in sys.modules:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/genai_synthea/run_02"
else:
    BASE = os.environ.get("RUN_BASE", "run_02")
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

F = pd.read_parquet(f"{BASE}/features_full/encounters_524.parquet")
fm = json.load(open(f"{BASE}/features_full/features_meta.json"))
meta = json.load(open(f"{BASE}/genai/meta.json"))
tab = pd.concat([pd.read_parquet(f"{BASE}/genai/{s}.parquet") for s in ("train", "val", "test")])
TAB_COLS = meta["num_cols"] + meta["cat_cols"] + meta["bin_cols"]
D = F.merge(tab[["hadm_id"] + [c for c in TAB_COLS if c not in F.columns]], on="hadm_id", how="left")
print(f"Stays {len(D):,} | PLOS {D.is_plos.mean():.1%} | split:", D.split.value_counts().to_dict())

In [ ]:
# Cell 2 - feature sets
B = fm["blocks"]
PROFILE, FLAGS = fm["profile"], ["lab_active", "med_active"]
def tab_X(d): return pd.get_dummies(d[TAB_COLS], columns=meta["cat_cols"], dtype=float)
SETS = {
    "A. tabular (notebooks 01-08)":            ("tab", None),
    "B. profile + vitals (10)":                ("cols", PROFILE),
    "C. B + conditions embedding":             ("cols", PROFILE + B["conditions"]),
    "D. B + procedures embedding":             ("cols", PROFILE + B["procedures"]),
    "E. B + notes embedding":                  ("cols", PROFILE + B["notes"]),
    "F. B + medications embedding":            ("cols", PROFILE + B["medications"]),
    "G. B + activity flags":                   ("cols", PROFILE + FLAGS),
    "H. full 524 vector":                      ("cols", fm["feat524"]),
    "I. full 524 + tabular":                   ("both", fm["feat524"]),
}
def X_of(d, kind, cols):
    if kind == "tab": return tab_X(d)
    if kind == "cols": return d[cols].astype(float)
    t = tab_X(d).reset_index(drop=True)
    return pd.concat([d[cols].astype(float).reset_index(drop=True), t.drop(columns=[c for c in t if c in cols])], axis=1)

In [ ]:
# Cell 3 - train on TRAIN, report AP / AUROC on VAL and TEST, paired patient bootstrap CI vs the tabular baseline
tr, va, te = (D[D.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
NB = int(os.environ.get("NBOOT", 1000)); rng = np.random.default_rng(0)
pats = te.subject_id.values; up = np.unique(pats); pos = {p: np.where(pats == p)[0] for p in up}
boots = [np.concatenate([pos[p] for p in rng.choice(up, len(up))]) for _ in range(NB)]
yte = te.is_plos.values
preds, rows = {}, []
for name, (kind, cols) in SETS.items():
    Xtr = X_of(tr, kind, cols); cols_ = Xtr.columns
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(Xtr, tr.is_plos)
    pv = m.predict_proba(X_of(va, kind, cols).reindex(columns=cols_, fill_value=0))[:, 1]
    pt = m.predict_proba(X_of(te, kind, cols).reindex(columns=cols_, fill_value=0))[:, 1]
    preds[name] = pt
    rows.append({"features": name, "n_features": Xtr.shape[1],
                 "AP_val": average_precision_score(va.is_plos, pv), "AP_test": average_precision_score(yte, pt),
                 "AUROC_test": roc_auc_score(yte, pt)})
base = preds["A. tabular (notebooks 01-08)"]
for r in rows:
    g = [average_precision_score(yte[i], preds[r["features"]][i]) - average_precision_score(yte[i], base[i])
         for i in boots if yte[i].sum() > 0]
    r["AP_gain_vs_A"], r["ci_low"], r["ci_high"] = r["AP_test"] - average_precision_score(yte, base), *np.percentile(g, [2.5, 97.5])
res = pd.DataFrame(rows).set_index("features")
res.to_csv(f"{BASE}/features_full/09a_feature_utility.csv")
print(f"Test PLOS prevalence (AP of a random model): {yte.mean():.3f}")
res.round(4)

In [ ]:
# Cell 4 - admission-safe check: age + demographics only vs the same plus during-stay blocks
SAFE = ["anchor_age", "gender_enc", "race_enc", "ethnicity_enc"]
STAY = {"notes": B["notes"], "medications": B["medications"] + ["med_active"],
        "procedures": B["procedures"], "vitals+labs": [c for c in PROFILE if c.startswith("vital_")] + ["has_vitals", "lab_active"]}
out = []
for name, cols in [("demographics only (admission-safe)", SAFE)] + [(f"+ {k} (during stay)", SAFE + v) for k, v in STAY.items()]:
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(tr[cols], tr.is_plos)
    out.append({"features": name, "AP_test": average_precision_score(yte, m.predict_proba(te[cols])[:, 1])})
print("How much of any gain comes from information recorded during the stay:")
pd.DataFrame(out).set_index("features").round(4)